# Sales Demand Forecasting - Exploratory Analysis

This notebook provides a step-by-step walkthrough of the data analysis and forecasting pipeline.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import sys, os

# Add parent directory to path
sys.path.insert(0, os.path.abspath('..'))

from src.data_cleaning import load_and_prepare, get_data_summary
from src.eda import create_kpi_metrics
from src.forecasting import run_forecast_pipeline
from src.business_insights import generate_insights

print('All imports successful!')

## 1. Load and Clean Data

In [ ]:
df, col_map, operations = load_and_prepare(file_path='../data/sample_sales_data.csv')

print('Dataset Shape:', df.shape)
print('\nCleaning Operations:')
for op in operations:
    print(f'  - {op}')

df.head()

## 2. Data Summary

In [ ]:
summary = get_data_summary(df)
print(f"Rows: {summary['rows']:,}")
print(f"Columns: {summary['columns']}")
print(f"Missing Values: {summary['total_missing']}")
print(f"Duplicates: {summary['duplicates']}")
print(f"Memory: {summary['memory_mb']} MB")
print(f"\nColumn Types:")
for col, dtype in summary['dtypes'].items():
    print(f"  {col}: {dtype}")

## 3. KPI Metrics

In [ ]:
metrics = create_kpi_metrics(df, col_map)
for k, v in metrics.items():
    print(f'{k}: {v:,.2f}' if isinstance(v, float) else f'{k}: {v}')

## 4. Revenue Trends

In [ ]:
date_col = col_map['date']
rev_col = col_map['revenue']

monthly = df.groupby(df[date_col].dt.to_period('M')).agg({rev_col: 'sum'}).reset_index()
monthly[date_col] = monthly[date_col].dt.to_timestamp()

fig = px.line(monthly, x=date_col, y=rev_col, title='Monthly Revenue Trend',
              template='plotly_white')
fig.show()

## 5. Category Analysis

In [ ]:
cat_col = col_map['category']
profit_col = col_map['profit']

cat_data = df.groupby(cat_col).agg({rev_col: 'sum', profit_col: 'sum'}).reset_index()
cat_data = cat_data.sort_values(rev_col, ascending=False)

fig = px.bar(cat_data, x=cat_col, y=[rev_col, profit_col],
             title='Category Performance', barmode='group',
             template='plotly_white')
fig.show()

## 6. Top 10 Products

In [ ]:
prod_col = col_map['product']
top_products = df.groupby(prod_col)[rev_col].sum().nlargest(10).reset_index()

fig = px.bar(top_products, x=rev_col, y=prod_col, orientation='h',
             title='Top 10 Products by Revenue', template='plotly_white')
fig.show()

## 7. Demand Forecasting

In [ ]:
results = run_forecast_pipeline(df, col_map, freq='M')
print(f"Success: {results['success']}")
print(f"Message: {results['message']}")

if results['success']:
    print(f"\nBaseline Metrics: {results['baseline_metrics']}")
    if results['model_metrics']:
        print(f"Model Metrics: {results['model_metrics']}")
    
    if results['future_forecast'] is not None:
        print(f"\nFuture Forecast:")
        print(results['future_forecast'].to_string(index=False))

## 8. Business Insights

In [ ]:
insights = generate_insights(df, col_map)
print(f'Generated {len(insights)} insights:\n')
for insight in insights:
    print(f"{insight['icon']} {insight['title']}")
    print(f"   {insight['detail']}\n")